In [0]:
%sql
-- ============================================================================
-- DDL GOLD: Creación de Esquema y Modelo Dimensional (Esquema Estrella)
-- ============================================================================
-- Script centralizado para la creación de infraestructura de la capa Gold
-- Ejecutar ANTES del proceso de modelado dimensional (04_Gold)
-- ============================================================================

-- 1. Crear esquema Gold si no existe
CREATE SCHEMA IF NOT EXISTS workspace.tp_dnrpa_gold
COMMENT 'Capa Gold - Modelo Dimensional (Esquema Estrella) para análisis de transferencias vehiculares';

In [0]:
%sql
-- ============================================================================
-- DIMENSIÓN: dim_marca
-- ============================================================================
-- Catálogo de marcas de vehículos
-- PK: automotor_marca_codigo

CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_marca (
  automotor_marca_codigo STRING NOT NULL COMMENT 'PK - Código único de la marca',
  automotor_marca_descripcion STRING COMMENT 'Nombre de la marca (normalizado)'
)
USING DELTA
COMMENT 'Dimensión de marcas de vehículos';

In [0]:
%sql
-- ============================================================================
-- DIMENSIÓN: dim_tipo_vehiculo
-- ============================================================================
-- Catálogo de tipos de vehículos
-- PK: automotor_tipo_codigo

CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_tipo_vehiculo (
  automotor_tipo_codigo STRING NOT NULL COMMENT 'PK - Código único del tipo de vehículo',
  automotor_tipo_descripcion STRING COMMENT 'Descripción del tipo de vehículo'
)
USING DELTA
COMMENT 'Dimensión de tipos de vehículos';

In [0]:
%sql
-- ============================================================================
-- DIMENSIÓN: dim_modelo
-- ============================================================================
-- Catálogo de modelos de vehículos
-- PK: automotor_modelo_codigo

CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_modelo (
  automotor_modelo_codigo STRING NOT NULL COMMENT 'PK - Código único del modelo',
  automotor_modelo_descripcion STRING COMMENT 'Nombre del modelo'
)
USING DELTA
COMMENT 'Dimensión de modelos de vehículos';

In [0]:
%sql
-- ============================================================================
-- DIMENSIÓN: dim_geografia
-- ============================================================================
-- Catálogo de registros seccionales y provincias
-- PK: registro_seccional_codigo

CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.dim_geografia (
  registro_seccional_codigo STRING NOT NULL COMMENT 'PK - Código único del registro seccional',
  registro_seccional_provincia STRING COMMENT 'Provincia del registro seccional'
)
USING DELTA
COMMENT 'Dimensión geográfica (seccionales y provincias)';

In [0]:
%sql
-- ============================================================================
-- TABLA DE HECHOS: fact_transferencias
-- ============================================================================
-- Transacciones de transferencias vehiculares con métricas y FKs a dimensiones
-- PK: id_tramite
-- FKs: automotor_marca_codigo, automotor_tipo_codigo, automotor_modelo_codigo, 
--      registro_seccional_codigo

CREATE TABLE IF NOT EXISTS workspace.tp_dnrpa_gold.fact_transferencias (
  id_tramite STRING NOT NULL COMMENT 'PK - Identificador único del trámite',
  tramite_fecha DATE COMMENT 'Fecha en que se realizó la transferencia',
  automotor_anio_modelo STRING COMMENT 'Año del modelo del vehículo (métrica)',
  automotor_marca_codigo STRING COMMENT 'FK - Código de marca (relaciona con dim_marca)',
  automotor_tipo_codigo STRING COMMENT 'FK - Código de tipo (relaciona con dim_tipo_vehiculo)',
  automotor_modelo_codigo STRING COMMENT 'FK - Código de modelo (relaciona con dim_modelo)',
  registro_seccional_codigo STRING COMMENT 'FK - Código de registro seccional (relaciona con dim_geografia)'
)
USING DELTA
COMMENT 'Tabla de hechos de transferencias vehiculares';

In [0]:
%sql
-- ============================================================================
-- VISTA ANALÍTICA: vw_transferencias_analitica
-- ============================================================================
-- Vista desnormalizada que une la tabla de hechos con todas sus dimensiones
-- Facilita consultas analíticas sin necesidad de escribir JOINs manualmente

CREATE OR REPLACE VIEW workspace.tp_dnrpa_gold.vw_transferencias_analitica
COMMENT 'Vista analítica desnormalizada de transferencias con dimensiones'
AS
SELECT
  -- Tabla de Hechos
  f.id_tramite,
  f.tramite_fecha,
  f.automotor_anio_modelo,
  
  -- Dimensión: Marca
  m.automotor_marca_codigo,
  m.automotor_marca_descripcion,
  
  -- Dimensión: Tipo de Vehículo
  tv.automotor_tipo_codigo,
  tv.automotor_tipo_descripcion,
  
  -- Dimensión: Modelo
  mo.automotor_modelo_codigo,
  mo.automotor_modelo_descripcion,
  
  -- Dimensión: Geografía
  g.registro_seccional_codigo,
  g.registro_seccional_provincia
  
FROM workspace.tp_dnrpa_gold.fact_transferencias f
LEFT JOIN workspace.tp_dnrpa_gold.dim_marca m
  ON f.automotor_marca_codigo = m.automotor_marca_codigo
LEFT JOIN workspace.tp_dnrpa_gold.dim_tipo_vehiculo tv
  ON f.automotor_tipo_codigo = tv.automotor_tipo_codigo
LEFT JOIN workspace.tp_dnrpa_gold.dim_modelo mo
  ON f.automotor_modelo_codigo = mo.automotor_modelo_codigo
LEFT JOIN workspace.tp_dnrpa_gold.dim_geografia g
  ON f.registro_seccional_codigo = g.registro_seccional_codigo;

## ✅ DDL Gold Completado

### Estructura Creada:

**Esquema:**
* `workspace.tp_dnrpa_gold` - Capa Gold con modelo dimensional

**Dimensiones (4 tablas):**
1. `dim_marca` - 700 marcas únicas
2. `dim_tipo_vehiculo` - 609 tipos únicos
3. `dim_modelo` - 10,389 modelos únicos
4. `dim_geografia` - 846 registros seccionales

**Tabla de Hechos:**
* `fact_transferencias` - 2.4M transacciones de transferencias

**Vista Analítica:**
* `vw_transferencias_analitica` - Modelo estrella desnormalizado

### Relaciones del Modelo:
```
fact_transferencias (PK: id_tramite)
  ├── FK: automotor_marca_codigo → dim_marca
  ├── FK: automotor_tipo_codigo → dim_tipo_vehiculo
  ├── FK: automotor_modelo_codigo → dim_modelo
  └── FK: registro_seccional_codigo → dim_geografia
```

### Uso de la Vista:
```sql
-- Ejemplo: Análisis de transferencias por marca y provincia
SELECT 
  automotor_marca_descripcion,
  registro_seccional_provincia,
  COUNT(*) as total_transferencias
FROM workspace.tp_dnrpa_gold.vw_transferencias_analitica
WHERE YEAR(tramite_fecha) = 2024
GROUP BY 1, 2
ORDER BY 3 DESC
LIMIT 10;
```